# 2. Fit the two lattice bases

Run **01_find_peaks_and_vectors.ipynb** first. This notebook reads its saved result, refines the origin and vectors, compares model residuals, and exports fitted vectors for **03_calibration.ipynb**.

Fitting and calibration are independent. This notebook works entirely in binned pixel coordinates.

Two original models are available: `parent` fits the two parent lattices; `explicit_moire` also enables the original explicit moiré intensity component in stage 3. Neither choice changes the archived algorithms.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

# Locate this checkout when launched from the project or an examples directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/moire_cdw").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
import json
from types import SimpleNamespace
import torch
from moire_cdw.fitting import (
    TwoLatticeMoireFitter, ExplicitMoireFitter, get_auto_init_from_moire,
    make_center_weight_mask, compute_fit_metrics, compare_vector_sets,
    plot_before_after_models,
)
from moire_cdw.results import save_fit_result
torch.set_num_threads(4)

## Fitting settings

`QUICK_RUN=True` uses three steps per stage to demonstrate the workflow. This is **not a converged analysis**. Set it to `False` to use the iteration counts from the original notebooks, then inspect the loss, residuals, and vectors.

In [ ]:
DETECTION_FILE = ROOT / "outputs/two_lattice/detection.npz"
OUTPUT = ROOT / "outputs/two_lattice/fit.npz"
MODEL = "parent"  # "parent" or "explicit_moire"
QUICK_RUN = True
DEVICE = "cpu"
BUILD_SETTINGS = dict(
    center_radius_px=9, spot_radius_px=5, origin_delta_px=6.0, uv_delta_px=4.0,
    lattice_order=6, center_intensity=1.0, spot_intensity=0.12,
    gaussian_bg_sigma=70.0, gaussian_bg_intensity=0.03, dc_bg_intensity=0.005,
    refine_center_template=False, refine_spot_templates=True,
    tv_weight=1e-4, boundary_px=7.0,
)
EXPLICIT_SETTINGS = dict(
    refine_moire_template=True, moire_intensity=0.02, moire_max_tile=15,
    moire_exclude_radius_px=2.0, moire_parent_maxlen_px=200.0, moire_round_decimals=3,
)
STAGE_SETTINGS = dict(lr1=2e-1, lr2=5e-2, lr3=2e-2, constraint_weight=1.0)

## Load the saved detection result

In [ ]:
if not DETECTION_FILE.exists():
    raise FileNotFoundError("Run 01_find_peaks_and_vectors.ipynb first.")
with np.load(DETECTION_FILE, allow_pickle=False) as saved:
    image = saved["image"].copy()
    peaks = saved["peaks"].copy()
    vector_indices = saved["vector_indices"].copy()
    sampling = saved["sampling"].copy()
    units = saved["units"].tolist()
    detection_metadata = json.loads(saved["metadata_json"].item())
auto_init = get_auto_init_from_moire(SimpleNamespace(peaks=peaks), vector_indices)
vectors = {key: auto_init[key] for key in ("origin_rc", "u1_rc", "v1_rc", "u2_rc", "v2_rc")}
print("Input:", detection_metadata["source_file"])
print("Initial vectors:", vectors)

## Build the model and preserve its initial state

In [ ]:
model_classes = {"parent": TwoLatticeMoireFitter, "explicit_moire": ExplicitMoireFitter}
fitter = model_classes[MODEL](image, device=DEVICE, dtype=torch.float32, loss_mode="log1p")
center_mask = make_center_weight_mask(image.shape, vectors["origin_rc"],
                                      radius_px=12, center_weight=0.20)
fitter.set_mask(center_mask)
build_settings = dict(BUILD_SETTINGS)
if MODEL == "explicit_moire":
    build_settings.update(EXPLICIT_SETTINGS)
fitter.build_from_auto_init(**vectors, peak_seed_rc=peaks[vector_indices[1:]], **build_settings)
initial_summary = fitter.summary()
initial_render = fitter.render_numpy()
fitter.plot(log_display=True, overlay=True)

## Fit in three stages

In [ ]:
steps = (3, 3, 3) if QUICK_RUN else ((40, 150, 100) if MODEL == "parent" else (40, 100, 100))
stage_settings = dict(STAGE_SETTINGS, stage1_steps=steps[0], stage2_steps=steps[1], stage3_steps=steps[2])
fitter.fit_staged(**stage_settings)
refined_summary = fitter.summary()
final_render = fitter.render_numpy()
fitter.plot_loss()

## Inspect the result

The metrics describe agreement with this image; improvement alone does not establish physical correctness. Both renders use the same normalized target and mask.

In [ ]:
print("Before:", compute_fit_metrics(fitter.image, initial_render, mask=center_mask))
print("After: ", compute_fit_metrics(fitter.image, final_render, mask=center_mask))
for key, before, after, delta, delta_norm in compare_vector_sets(initial_summary, refined_summary):
    print(f"{key}: {before} -> {after}; change = {delta_norm:.4f} px")
plot_before_after_models(fitter.image, initial_render, final_render)
fitter.plot(log_display=True, overlay=True)

## Export fitted vectors for calibration

The NPZ includes binned reciprocal sampling, input provenance, model settings, and loss history. Calibration can read it in a separate kernel. Rerunning replaces the chosen output file.

In [ ]:
save_fit_result(
    OUTPUT, refined_summary, sampling=sampling, units=units,
    source_file=detection_metadata["source_file"], bin_factor=detection_metadata["bin_factor"],
    image_shape=image.shape, loss_history=fitter.loss_history, initial_summary=initial_summary,
    fit_settings=dict(model=MODEL, quick_run=QUICK_RUN, device=DEVICE, loss_mode="log1p",
                      build=build_settings, stages=stage_settings,
                      center_mask=dict(radius_px=12, center_weight=0.20),
                      detection=detection_metadata),
)
print("Saved:", OUTPUT)

Next: open **03_calibration.ipynb**. Calibration does not rerun or modify fitting.